In [0]:
"""
04_material_dimension.py

Material Dimension (SCD Type 2)

Source:
    material_events

Target:
    material_dimension

Author:
Sumanth Vempalle

Version:
2.2.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    lit,
)

# ============================================================
# Material Source View
# ============================================================

@dlt.view(
    name="material_dimension_source",
    comment="Source view for Material Dimension."
)
def material_dimension_source():

    return (

        spark.readStream.table(
            "material_events"
        )

        .select(

            col("material_number"),

            col("batch_number"),

            col("supplier"),

            col("product_code"),

            col("product_name"),

            col("family"),

            # Placeholder attributes
            lit("Manufacturing Material").alias(
                "material_type"
            ),

            lit("ACTIVE").alias(
                "material_status"
            ),

            col("event_timestamp").alias(
                "last_updated"
            ),

        )

        .filter(
            col("material_number").isNotNull()
        )

        .dropDuplicates(
            [
                "material_number",
                "last_updated"
            ]
        )

    )


# ============================================================
# Target Streaming Table
# ============================================================

dlt.create_streaming_table(

    name="material_dimension",

    comment="Material Dimension (SCD Type 2)."

)


# ============================================================
# AUTO CDC FLOW
# ============================================================

dlt.create_auto_cdc_flow(

    target="material_dimension",

    source="material_dimension_source",

    keys=[
        "material_number",
    ],

    sequence_by="last_updated",

    stored_as_scd_type=2,

    track_history_except_column_list=[
        "last_updated",
    ],

)